# Security Cam - Micro Classificatore per ESP32-S3

Addestramento di un classificatore leggero (~100 KB) che gira **direttamente sull'ESP32-S3** per filtrare i falsi positivi del PIR prima dell'upload su Firebase.

**Architettura**: MobileNetV2 (alpha=0.25) con transfer learning
- Input: 96×96×3 RGB (uint8)
- Output: 3 sigmoid [person, cat, dog]
- Dimensione: ~100-200 KB (int8 quantizzato)
- Tempo inferenza ESP32-S3: ~1-2 secondi


## 1. Dipendenze

In [ ]:
!pip install -q pycocotools

import os, json, shutil, random
import numpy as np
from collections import Counter
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("\n Pronto")

TensorFlow: 2.19.0
GPU: True

✅ Pronto


## 2. Download COCO 2017

In [ ]:
print("Download immagini COCO val2017 (~1GB)...")
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip
!rm val2017.zip

print("Download annotazioni...")
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip
!rm annotations_trainval2017.zip

print("\n COCO scaricato")

📥 Download immagini COCO val2017 (~1GB)...
val2017.zip         100%[===================>] 777.80M  55.3MB/s    in 14s     

📥 Download annotazioni...
annotations_trainva 100%[===================>] 241.19M  44.0MB/s    in 5.8s    

✅ COCO scaricato


## 3. Preparazione dataset

Per il classificatore ESP servono **immagini intere** con label multi-label (non bounding box).
Per ogni immagine sappiamo: contiene una persona? un gatto? un cane? (puo' contenerne piu' di uno).

Aggiungiamo anche **immagini negative** (senza nessuna delle 3 categorie) per insegnare al modello a dire "qui non c'e' niente di interessante".

In [ ]:
print("Preparazione dataset classificazione...")

COCO_CAT_IDS = {1: 0, 17: 1, 18: 2}  # person->0, cat->1, dog->2
CLASS_NAMES = ["person", "cat", "dog"]

with open("annotations/instances_val2017.json", "r") as f:
    coco = json.load(f)

img_info = {img["id"]: img for img in coco["images"]}

# Annotazioni delle 3 categorie
target_anns = [a for a in coco["annotations"] if a["category_id"] in COCO_CAT_IDS]
target_img_ids = set(a["image_id"] for a in target_anns)
all_img_ids = set(img["id"] for img in coco["images"])
negative_img_ids = all_img_ids - target_img_ids

print(f"  Immagini con person/cat/dog: {len(target_img_ids)}")
print(f"  Immagini negative:           {len(negative_img_ids)}")

# Per ogni immagine target, determina quali classi contiene
img_labels = {}
for ann in target_anns:
    img_id = ann["image_id"]
    cls_idx = COCO_CAT_IDS[ann["category_id"]]
    if img_id not in img_labels:
        img_labels[img_id] = [0, 0, 0]
    img_labels[img_id][cls_idx] = 1

cat_counts = Counter()
for labels in img_labels.values():
    for i, v in enumerate(labels):
        if v == 1:
            cat_counts[CLASS_NAMES[i]] += 1
for name, count in cat_counts.most_common():
    print(f"    {name}: {count} immagini")

# Split 80/20
pos_list = sorted(target_img_ids)
random.seed(42)
random.shuffle(pos_list)
split_pos = int(len(pos_list) * 0.8)

neg_list = sorted(negative_img_ids)
random.shuffle(neg_list)
# Limitiamo i negativi a ~30% del totale per bilanciare
n_neg = int(len(pos_list) * 0.35)

train_pos = set(pos_list[:split_pos])
val_pos = set(pos_list[split_pos:])
train_neg = set(neg_list[:int(n_neg * 0.8)])
val_neg = set(neg_list[int(n_neg * 0.8):n_neg])

print(f"\n Split:")
print(f"  Train: {len(train_pos)} positivi + {len(train_neg)} negativi = {len(train_pos)+len(train_neg)}")
print(f"  Val:   {len(val_pos)} positivi + {len(val_neg)} negativi = {len(val_pos)+len(val_neg)}")

# Copia immagini nelle directory
dataset_info = {}  # path -> label

for split_name, pos_ids, neg_ids in [
    ("train", train_pos, train_neg),
    ("val", val_pos, val_neg)
]:
    out_dir = f"clf_dataset/{split_name}"
    os.makedirs(out_dir, exist_ok=True)

    for img_id in pos_ids:
        info = img_info[img_id]
        src = f"val2017/{info['file_name']}"
        if os.path.exists(src):
            dst = f"{out_dir}/{info['file_name']}"
            shutil.copy2(src, dst)
            dataset_info[dst] = img_labels[img_id]

    for img_id in neg_ids:
        info = img_info[img_id]
        src = f"val2017/{info['file_name']}"
        if os.path.exists(src):
            dst = f"{out_dir}/{info['file_name']}"
            shutil.copy2(src, dst)
            dataset_info[dst] = [0, 0, 0]

with open("clf_dataset/labels.json", "w") as f:
    json.dump(dataset_info, f)

# Libera spazio
!rm -rf val2017/ annotations/

print("\n Dataset pronto")

Preparazione dataset classificazione...
  Immagini con person/cat/dog: 2945
  Immagini negative:           2055
    person: 2693 immagini
    cat: 184 immagini
    dog: 177 immagini

 Split:
  Train: 2356 positivi + 824 negativi = 3180
  Val:   589 positivi + 206 negativi = 795

 Dataset pronto


## 4. Caricamento e preprocessing

Ridimensioniamo tutte le immagini a **96×96** pixel.
Questo e' l'input che l'ESP mandara' al modello: cattura un frame VGA (640×480), lo ridimensiona a 96×96 in software, e lo passa al classificatore.

In [ ]:
from PIL import Image as PILImage

IMG_SIZE = 96

def load_split(split_name):
    images = []
    labels = []
    with open("clf_dataset/labels.json", "r") as f:
        all_labels = json.load(f)

    for path, label in all_labels.items():
        if f"/{split_name}/" not in path:
            continue
        if not os.path.exists(path):
            continue
        try:
            img = PILImage.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
            images.append(np.array(img, dtype=np.float32) / 255.0)
            labels.append(label)
        except:
            continue
    return np.array(images), np.array(labels, dtype=np.float32)

print(" Caricamento immagini (96×96)...")
X_train, y_train = load_split("train")
X_val, y_val = load_split("val")

print(f"  Train: {X_train.shape}")
print(f"  Val:   {X_val.shape}")

print(f"\n  Distribuzione classi (train):")
for i, name in enumerate(CLASS_NAMES):
    pos = int(y_train[:, i].sum())
    neg = len(y_train) - pos
    print(f"    {name}: {pos} positivi, {neg} negativi ({pos/len(y_train)*100:.1f}%)")

all_neg = int((y_train.sum(axis=1) == 0).sum())
print(f"    [nessuna]: {all_neg} ({all_neg/len(y_train)*100:.1f}%)")

 Caricamento immagini (96×96)...
  Train: (3180, 96, 96, 3)
  Val:   (795, 96, 96, 3)

  Distribuzione classi (train):
    person: 2151 positivi, 1029 negativi (67.6%)
    cat: 149 positivi, 3031 negativi (4.7%)
    dog: 140 positivi, 3040 negativi (4.4%)
    [nessuna]: 824 (25.9%)


## 5. Costruzione modello

**MobileNetV2 (alpha=0.25)**: la variante piu' piccola della famiglia MobileNetV2.

- `alpha=0.25` → usa solo 1/4 dei filtri rispetto alla versione standard
- `include_top=False` → togliamo la testa originale (1000 classi ImageNet)
- `weights="imagenet"` → backbone pre-trained (**transfer learning**)
- Congeliamo il backbone e addestriamo solo la testa custom

La testa custom:
- `GlobalAveragePooling2D` → riduce i feature maps a un vettore
- `Dropout(0.2)` → regolarizzazione contro overfitting
- `Dense(32, relu)` → layer intermedio
- `Dense(3, sigmoid)` → 3 output indipendenti (multi-label)

`sigmoid` invece di `softmax` perche' le classi non sono mutuamente esclusive: una foto puo' contenere sia una persona che un cane.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print(" Costruzione modello...")

# Backbone pre-trained (congelato)
base_model = MobileNetV2(
    alpha=0.35,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

# Testa custom (trainabile)
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
x = Dense(32, activation="relu")(x)
x = Dense(3, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=x)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)

total = model.count_params()
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
frozen = total - trainable

print(f"\n  Parametri totali:     {total:,}")
print(f"  Trainabili (testa):   {trainable:,}")
print(f"  Congelati (backbone): {frozen:,}")
print(f"  Ratio:                {trainable/total*100:.1f}% trainabile")

model.summary()

 Costruzione modello...

  Parametri totali:     451,299
  Trainabili (testa):   41,091
  Congelati (backbone): 410,208
  Ratio:                9.1% trainabile


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 48, 48,    │        432 │ input_layer_1[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 48, 48,    │         64 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 48, 48,    │          0 │ bn_Conv1[0][0]    │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │        144 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 48, 48,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48, 8) │        128 │ expanded_conv_de… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 48, 48, 8) │         32 │ expanded_conv_pr… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 48, 48,    │        384 │ expanded_conv_pr… │
│ (Conv2D)            │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 48, 48,    │        192 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 48, 48,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 49, 49,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 24, 24,    │        432 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │        192 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 24, 24,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 24, 24, 8) │        384 │ block_1_depthwis

 Total params: 451,299 (1.72 MB)

 Trainable params: 41,091 (160.51 KB)

 Non-trainable params: 410,208 (1.56 MB)

## 6. Training

- **30 epoch** con early stopping (si ferma se la val_loss non migliora per 5 epoch)
- **binary_crossentropy** come loss (standard per multi-label)
- **batch_size=32**

⏱️ ~3-5 minuti

In [ ]:
print(" Training...")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    ],
    verbose=1
)

best_epoch = np.argmin(history.history['val_loss']) + 1
print(f"\n Training completato! Migliore epoca: {best_epoch}")

 Training...
Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 47s 280ms/step - binary_accuracy: 0.8791 - loss: 0.2913 - val_binary_accuracy: 0.9023 - val_loss: 0.2531
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - binary_accuracy: 0.9034 - loss: 0.2305 - val_binary_accuracy: 0.9023 - val_loss: 0.2452
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - binary_accuracy: 0.9138 - loss: 0.2094 - val_binary_accuracy: 0.9015 - val_loss: 0.2396
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - binary_accuracy: 0.9210 - loss: 0.1923 - val_binary_accuracy: 0.8998 - val_loss: 0.2455
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - binary_accuracy: 0.9274 - loss: 0.1782 - val_binary_accuracy: 0.9023 - val_loss: 0.2469
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - binary_accuracy: 0.9308 - loss: 0.1667 - val_binary_accuracy: 0.9031 - val_loss: 0.2505
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - binary_accuracy: 0.9339 - loss: 0.1576 - val_binary_accuracy: 0.8985 - v

## 7. Valutazione

In [ ]:
import matplotlib.pyplot as plt

y_pred = model.predict(X_val, verbose=0)

print(f"{'='*55}")
print(f"  RISULTATI MICRO CLASSIFICATORE ESP32")
print(f"{'='*55}")

for i, name in enumerate(CLASS_NAMES):
    pred_bin = (y_pred[:, i] > 0.5).astype(int)
    true_bin = y_val[:, i].astype(int)

    tp = int(((pred_bin == 1) & (true_bin == 1)).sum())
    fp = int(((pred_bin == 1) & (true_bin == 0)).sum())
    fn = int(((pred_bin == 0) & (true_bin == 1)).sum())
    tn = int(((pred_bin == 0) & (true_bin == 0)).sum())

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    acc  = (tp + tn) / len(true_bin)
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    print(f"\n  {name}:")
    print(f"    Accuracy:  {acc:.3f}")
    print(f"    Precision: {prec:.3f}  (dei rilevamenti, quanti sono giusti)")
    print(f"    Recall:    {rec:.3f}  (degli oggetti reali, quanti trovati)")
    print(f"    F1-score:  {f1:.3f}")
    print(f"    TP={tp} FP={fp} FN={fn} TN={tn}")

# Valutazione "filtro PIR": quanti upload eviteremmo?
any_pred = (y_pred.max(axis=1) > 0.5)
any_true = (y_val.max(axis=1) > 0.5)
would_upload = any_pred.sum()
should_upload = any_true.sum()
total_triggers = len(y_val)
saved = total_triggers - would_upload

print(f"\n{'='*55}")
print(f"  IMPATTO COME FILTRO PIR:")
print(f"    Trigger PIR simulati:    {total_triggers}")
print(f"    Upload effettuati:       {int(would_upload)} ({would_upload/total_triggers*100:.0f}%)")
print(f"    Upload risparmiati:      {int(saved)} ({saved/total_triggers*100:.0f}%)")
print(f"    Upload necessari persi:  {int(((~any_pred) & any_true).sum())}")
print(f"{'='*55}")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='Train')
ax1.plot(history.history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)

ax2.plot(history.history['binary_accuracy'], label='Train')
ax2.plot(history.history['val_binary_accuracy'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)

plt.suptitle("Micro Classificatore ESP32 - Training")
plt.tight_layout()
plt.savefig("micro_curves.png", dpi=150)
plt.show()

## 8. Export TFLite con quantizzazione int8

Fondamentale per ESP32:
- **int8 e' ~4x piu' veloce** su microcontroller
- Input/output in **uint8** (0-255): l'immagine dalla camera e' gia' uint8, nessuna conversione necessaria sull'ESP

**NOTA IMPORTANTE**
Alla fine non è stato eseguito perchè la quantizzazione portava a un errore su un certo indirizzo che significava memoria non inizializzata, probabilmente un puntatore interno ai tensori intermendi era corrotto.

In [ ]:
print(" Quantizzazione int8...")

def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

os.makedirs("output", exist_ok=True)
tflite_path = "output/security_cam_filter.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f"\n Modello TFLite: {size_kb:.0f} KB")

# Verifica
interp = tf.lite.Interpreter(model_path=tflite_path)
interp.allocate_tensors()
inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]
print(f"  Input:  shape={inp_det['shape']} dtype={inp_det['dtype']}")
print(f"  Output: shape={out_det['shape']} dtype={out_det['dtype']}")

# Test
test_img = (X_val[0:1] * 255).astype(np.uint8)
interp.set_tensor(inp_det['index'], test_img)
interp.invoke()
result = interp.get_tensor(out_det['index'])[0].astype(float) / 255.0
true_label = y_val[0]
print(f"\n  Test inferenza TFLite int8:")
print(f"    Predetto: person={result[0]:.2f} cat={result[1]:.2f} dog={result[2]:.2f}")
print(f"    Reale:    person={true_label[0]:.0f}    cat={true_label[1]:.0f}    dog={true_label[2]:.0f}")

## 9. Conversione in header C


Il modello viene incluso nel firmware come array di byte in un file `.h`.

```c
// Il compilatore mette questo array in flash
const unsigned char micro_model_data[] = { 0x1c, 0x00, ... };
```

A runtime il firmware lo legge dalla flash e lo passa a TFLite Micro.

In [ ]:
print(" Generazione header C...")

with open(tflite_path, "rb") as f:
    model_bytes = f.read()

header = []
header.append('// ============================================================')
header.append('// Micro classificatore per ESP32-S3 - Security Cam')
header.append('// Generato dal notebook di training')
header.append(f'// Dimensione: {len(model_bytes)} bytes ({len(model_bytes)/1024:.0f} KB)')
header.append(f'// Input:  96x96x3 uint8 RGB')
header.append(f'// Output: 3 uint8 [person, cat, dog] (0-255 -> 0.0-1.0)')
header.append('// ============================================================')
header.append('')
header.append('#ifndef MICRO_MODEL_DATA_H')
header.append('#define MICRO_MODEL_DATA_H')
header.append('')
header.append('#include <stdint.h>')
header.append('')
header.append(f'#define MICRO_MODEL_SIZE {len(model_bytes)}')
header.append(f'#define MICRO_INPUT_SIZE 96')
header.append(f'#define MICRO_NUM_CLASSES 3')
header.append('')
header.append('// Nomi delle classi (in ordine)')
header.append('// 0: person, 1: cat, 2: dog')
header.append('')
header.append('alignas(8) const unsigned char micro_model_data[] = {')

# Scrivi bytes, 16 per riga
for i in range(0, len(model_bytes), 16):
    chunk = model_bytes[i:i+16]
    hex_str = ', '.join(f'0x{b:02x}' for b in chunk)
    comma = ',' if i + 16 < len(model_bytes) else ''
    header.append(f'    {hex_str}{comma}')

header.append('};')
header.append('')
header.append('#endif // MICRO_MODEL_DATA_H')
header.append('')

header_text = '\n'.join(header)
header_path = "output/micro_model_data.h"
with open(header_path, "w") as f:
    f.write(header_text)

size_h = os.path.getsize(header_path) / 1024
print(f"Header C: {header_path} ({size_h:.0f} KB)")
print(f"   → copia in main/include/ del progetto ESP-IDF")

 Generazione header C...
Header C: output/micro_model_data.h (4021 KB)
   → copia in main/include/ del progetto ESP-IDF


## 11. Download

| File | Dove va |
|---|---|
| `micro_model_data.h` | `main/include/` del progetto ESP-IDF |
| `security_cam_filter.tflite` | Opzionale (per test standalone) |

In [ ]:
from google.colab import files

print("Download...\n")

if os.path.exists("output/micro_model_data.h"):
    size = os.path.getsize("output/micro_model_data.h") / 1024
    print(f"  micro_model_data.h ({size:.0f} KB)")
    files.download("output/micro_model_data.h")

if os.path.exists("output/security_cam_filter.tflite"):
    size = os.path.getsize("output/security_cam_filter.tflite") / 1024
    print(f"  security_cam_filter.tflite ({size:.0f} KB)")
    files.download("output/security_cam_filter.tflite")

for img in ["micro_curves.png", "output/micro_test.png"]:
    if os.path.exists(img):
        files.download(img)

print()
print("=" * 55)
print("   FATTO!")
print("=" * 55)
print()
print("  Metti micro_model_data.h in:")
print("  main/include/micro_model_data.h")
print()
print("  Poi aggiungi esp-tflite-micro al firmware")
print("  e crea tflite_classifier.c per usarlo")
print("=" * 55)